In [1]:
%matplotlib qt
import mne
import os.path as op
import numpy as np
import matplotlib.pyplot as plt

# Read sample data

In [2]:
data_path = '~/data/sample/'
raw_file = op.join(data_path, 'sample_audvis_raw.fif')
raw = mne.io.read_raw_fif(raw_file, preload=False)
raw.pick(['eeg', 'eog', 'stim']).load_data()
raw.info

Opening raw data file ~/data/sample/sample_audvis_raw.fif...
    Read a total of 3 projection items:
        PCA-v1 (1 x 102)  idle
        PCA-v2 (1 x 102)  idle
        PCA-v3 (1 x 102)  idle
    Range : 25800 ... 192599 =     42.956 ...   320.670 secs
Ready.
Removing projector <Projection | PCA-v1, active : False, n_channels : 102>
Removing projector <Projection | PCA-v2, active : False, n_channels : 102>
Removing projector <Projection | PCA-v3, active : False, n_channels : 102>
Reading 0 ... 166799  =      0.000 ...   277.714 secs...


<Info | 20 non-empty values
 acq_pars: ACQch001 110113 ACQch002 110112 ACQch003 110111 ACQch004 110122 ...
 bads: 1 items (EEG 053)
 ch_names: STI 001, STI 002, STI 003, STI 004, STI 005, STI 006, STI 014, ...
 chs: 9 Stimulus, 60 EEG, 1 EOG
 custom_ref_applied: False
 description: acquisition (megacq) VectorView system at NMR-MGH
 dev_head_t: MEG device -> head transform
 dig: 146 items (3 Cardinal, 4 HPI, 61 EEG, 78 Extra)
 events: 1 item (list)
 experimenter: MEG
 file_id: 4 items (dict)
 highpass: 0.1 Hz
 hpi_meas: 1 item (list)
 hpi_results: 1 item (list)
 lowpass: 172.2 Hz
 meas_date: 2002-12-03 19:01:10 UTC
 meas_id: 4 items (dict)
 nchan: 70
 proj_id: 1 item (ndarray)
 proj_name: test
 projs: []
 sfreq: 600.6 Hz
>

# Visually inspect data for bad channels and spans

In [ ]:
# Visually inspect raw data and mark bad channels and span
raw.plot(n_channels=len(raw))

In [3]:
# Interpolate bad channel data
raw.interpolate_bads()

# raw.plot(n_channels=len(raw))

Interpolating bad channels
    Automatic origin fit: head of radius 91.0 mm
Computing interpolation matrix from 59 sensor positions
Interpolating 1 sensors


<Raw | sample_audvis_raw.fif, 70 x 166800 (277.7 s), ~92.0 MB, data loaded>

# Creating the average reference as projector

In [4]:
raw.set_eeg_reference('average', projection=True)
# print(raw.info['projs'])

# for title, proj in zip(['Original', 'Average'], [False, True]):
#     fig1 = raw.plot(proj=proj, n_channels=len(raw))
#     # make room for title
#     fig1.subplots_adjust(top=0.9)
#     fig1.suptitle('{} reference'.format(title), size='xx-large', weight='bold')

Adding average EEG reference projection.
1 projection items deactivated
Average reference projection was added, but has not been applied yet. Use the apply_proj method to apply it.


<Raw | sample_audvis_raw.fif, 70 x 166800 (277.7 s), ~92.1 MB, data loaded>

# Channel locations

In [ ]:
# fig2 = raw.plot_sensors(show_names=True)
# raw.plot_sensors('3d')

# Low-frequency drifts

In [ ]:
# fig3 = raw.plot(duration=60, n_channels=len(raw), remove_dc=False)

# raw.filter(l_freq=0.1, h_freq=None)
# fig4 = raw.plot(duration=60, n_channels=len(raw), remove_dc=False)

# Power line noise

In [ ]:
# fig5 = raw.plot_psd(tmax=np.inf, fmax=250, average=True)
# freqs = fig5.axes[0].lines[-1].get_xdata()
# psds = fig5.axes[0].lines[-1].get_ydata()
# for freq in (60, 120, 180, 240):
#     idx = np.searchsorted(freqs, freq)
#     fig5.axes[0].arrow(x=freqs[idx], y=psds[idx] + 18, dx=0, dy=-12, color='red',
#              width=0.1, head_width=3, length_includes_head=True)

# Use ICA to remove occular artifacts

In [ ]:
# eog_evoked = mne.preprocessing.create_eog_epochs(raw).average()
# eog_evoked.apply_baseline(baseline=(None, -0.2))
# fig6 = eog_evoked.plot_joint()

In [5]:
# Filtering to remove slow drifts
filt_raw_ica = raw.copy().filter(l_freq=1, h_freq=None)

# Fitting the ICA solution
ica = mne.preprocessing.ICA(n_components=15, max_iter='auto', random_state=97)
ica.fit(filt_raw_ica)
ica

Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 1983 samples (3.302 sec)

Fitting ICA to data using 60 channels (please be patient, this may take a while)


/opt/anaconda3/envs/mne/lib/python3.9/site-packages/mne/utils/check.py:92: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  LooseVersion(library.__version__) < LooseVersion(min_version):
/opt/anaconda3/envs/mne/lib/python3.9/site-packages/mne/utils/check.py:92: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  LooseVersion(library.__version__) < LooseVersion(min_version):


Selecting by number: 15 components
Fitting ICA took 3.0s.


Method,fastica
Fit,38 iterations on raw data (166800 samples)
ICA components,15
Explained variance,95.9 %
Available PCA components,60
Channel types,eeg
ICA components marked for exclusion,—


In [ ]:
# Plotting the ICA solution
fig7 = ica.plot_sources(raw, show_scrollbars=False)

# Visualize the scalp field distribution of ICA components
fig8 = ica.plot_components()

In [ ]:
# Additional visualization to check exclusion criteria
fig9 = ica.plot_overlay(raw, exclude=[0, 1])

In [6]:
# Manual selection of ICA components to be excluded
ica.exclude = [0, 1]

# Reconstruction of raw data after ICA exclusion
reconst_raw = raw.copy()
ica.apply(reconst_raw)

# Visualize repair
fig10 = raw.plot(n_channels=len(raw), show_scrollbars=False)
fig11 = reconst_raw.plot(n_channels=len(raw), show_scrollbars=False)

Applying ICA to Raw instance
    Transforming to ICA space (15 components)
    Zeroing out 2 ICA components
    Projecting back using 60 PCA components
Using matplotlib as 2D backend.


qt.qpa.drawing: Layer-backing can not be explicitly controlled on 10.14 when built against the 10.14 SDK


Channels marked as bad: none
Channels marked as bad: none


In [7]:
# Export preprocessed data to a new .fif file
reconst_raw_file = op.join(data_path, 'sample_audvis_eeg.fif')
reconst_raw.save(reconst_raw_file, overwrite=True)

Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Writing /Users/creator/data/sample/sample_audvis_eeg.fif
Closing /Users/creator/data/sample/sample_audvis_eeg.fif
[done]
